### Tarea 2 -> IA Generativa Aplicada a las Ciencias Sociales y la Gestión Pública

La base de datos a utilizar será diferente a la presentada en la Tarea 1 debido a que dicha tarea fue realizada con una base de datos que no es pública y fue compartida con fines académicos. Por ese motivo, se utilizara una fuente de datos en linea de Sunat CONSULTA tu RUC: https://e-consultaruc.sunat.gob.pe/

Esta base de datos, muestra en base a algun documento de identificación como DNI o RUC; el estado de actividad de dicho documento. De esta forma es una buena idea de interés verificar si es que tenemos una lista de empresas o personas naturales, junto a un grupo de dni's o rucs, para verificar si se encuentra operando en base a lo suscrito por SUNAT. Esto puede ser muy útil si es que se busca realizar algun tipo de verificación con las empresas, sobre todo si es que es a gran escala de empresas. Para fines de investigación o simplemente validar una lista de proveedores o contrapartes. El código permite automatizar este proceso para cualquier número de empresas, extrayendo el estado y condición del contribuyente de forma sistemática.

1. Renato Christian Gates Rojas (20181179)

Información a utilizar: RUCS

Los rucs son de las empresas más conocidas del país:

### Empresas y RUCS

| Empresa | RUC |  |
|---|---|---|
| `Banco de Crédito del Perú (BCP)` | 20100047218
| `Alicorp S.A.A.` | 20100055237
| `Unión de Cervecerías Peruanas Backus y Johnston S.A.A.` | 20100113610
| `Interbank (Banco Internacional del Perú S.A.A.)` | 20100053455
| `Ferreyros S.A.` | 20100028698
| `Supermercados Peruanos S.A. (Plaza Vea/Vivanda)` | 20100070970
| `Homecenters Peruanos S.A. (Promart)` | 20536557858
| `Telefónica del Perú S.A.A` | 20100017491
| `Nestlé Perú S.A.` | 20263322496
| `Latam Airlines Perú S.A.` | 20341841357
| `Cervecería San Juan S.A.` | 20128915711
| `Compañía Minera Antamina S.A.` | 20330262428
| `Cementos Pacasmayo S.A.A.` | 20419387658
| `Saga Falabella S.A.` | 20100128056
| `Tiendas Ripley S.A.` | 20337564373
| `Yo` | 10724680025

In [64]:
import sys
!{sys.executable} -m pip install --quiet selenium webdriver-manager openpyxl pandas

In [65]:
!apt-get update -qq
!apt-get install -y software-properties-common wget gnupg -qq
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update -qq
!apt-get install -y google-chrome-stable -qq

W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:2
W: Target Packages (main/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:2
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:3
W: Target Packages (main/binary-all/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt/sources.list.d/google-chrome.list:3
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.li

In [66]:
import pandas as pd
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from google.colab import drive

In [67]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [68]:
RUTA_DATA = '/content/drive/MyDrive/IA generativa/Tarea 2/'
RUTA_DATA

'/content/drive/MyDrive/IA generativa/Tarea 2/'

In [69]:
pagina = "https://e-consultaruc.sunat.gob.pe/cl-ti-itmrconsruc/FrameCriterioBusquedaWeb.jsp"

rucs = [
    "20100047218", "20100055237", "20100113610", "20100053455",
    "20100028698", "20100070970", "20536557858", "20100017491",
    "20263322496", "20341841357", "20128915711", "20330262428",
    "20419387658", "20100128056", "20337564373"
]

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--remote-debugging-port=9222')
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.93 Safari/537.36")
chrome_options.binary_location = '/usr/bin/google-chrome'

resultados = []

for empresa in rucs:
    driver = None
    try:
        service = Service(ChromeDriverManager().install())
        driver  = webdriver.Chrome(service=service, options=chrome_options)

        driver.get(pagina)
#30 segundos porque demora mucho en cargar a veces.
        wait = WebDriverWait(driver, 30)
        wait.until(EC.presence_of_element_located((By.ID, "txtRuc")))

        driver.find_element(By.ID, "txtRuc").send_keys(empresa)
        driver.find_element(By.ID, "btnAceptar").click()

        wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/div/div[2]/div/div[3]/div[2]/div[5]/div/div[2]/p')))

        estado_actual    = driver.find_element(By.XPATH, '/html/body/div/div[2]/div/div[3]/div[2]/div[5]/div/div[2]/p').text.strip()
        condicion_actual = driver.find_element(By.XPATH, '/html/body/div/div[2]/div/div[3]/div[2]/div[6]/div/div[2]/p').text.strip()

        resultados.append({
            "RUC"      : empresa,
            "Estado"   : estado_actual,
            "Condición": condicion_actual
        })

        print(f"RUC: {empresa} | Estado: {estado_actual} | Condición: {condicion_actual}")

    except Exception as e:
        nodata.append(empresa)
        print(f"Sin datos para RUC {empresa}: {e}")

    finally:
        if driver:
            driver.quit()

report_df = pd.DataFrame(resultados)
print("\n--- Resultados ---")
print(report_df)

path_output = RUTA_DATA + 'Scrapping_SUNAT'
os.makedirs(path_output, exist_ok=True)
output_path = os.path.join(path_output, "Estado_RUC_SUNAT.xlsx")

report_df.to_excel(output_path, index=False, sheet_name="Resultados")

workbook     = load_workbook(output_path)
sheet        = workbook.active
header_fill  = PatternFill(start_color="006400", end_color="006400", fill_type="solid")
header_font  = Font(color="FFFFFF", bold=True)

for cell in sheet[1]:
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")

for column in sheet.columns:
    max_length    = 0
    column_letter = column[0].column_letter
    for cell in column:
        try:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        except:
            pass
    sheet.column_dimensions[column_letter].width = max_length + 2

workbook.save(output_path)

# --- cerramos el browser como en clase ---
driver.quit()
print("Browser closed.")


RUC: 20100047218 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100055237 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100113610 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100053455 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100028698 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100070970 | Estado: ACTIVO | Condición: HABIDO
RUC: 20536557858 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100017491 | Estado: ACTIVO | Condición: HABIDO
RUC: 20263322496 | Estado: ACTIVO | Condición: HABIDO
RUC: 20341841357 | Estado: ACTIVO | Condición: HABIDO
RUC: 20128915711 | Estado: ACTIVO | Condición: HABIDO
RUC: 20330262428 | Estado: ACTIVO | Condición: HABIDO
RUC: 20419387658 | Estado: ACTIVO | Condición: HABIDO
RUC: 20100128056 | Estado: ACTIVO | Condición: HABIDO


RUC: 20337564373 | Estado: ACTIVO | Condición: HABIDO

--- Resultados ---
            RUC  Estado Condición
0   20100047218  ACTIVO    HABIDO
1   20100055237  ACTIVO    HABIDO
2   20100113610  ACTIVO    HABIDO
3   20100053455  ACTIVO    HABIDO
4   20100028698  ACTIVO    HABIDO
5   20100070970  ACTIVO    HABIDO
6   20536557858  ACTIVO    HABIDO
7   20100017491  ACTIVO    HABIDO
8   20263322496  ACTIVO    HABIDO
9   20341841357  ACTIVO    HABIDO
10  20128915711  ACTIVO    HABIDO
11  20330262428  ACTIVO    HABIDO
12  20419387658  ACTIVO    HABIDO
13  20100128056  ACTIVO    HABIDO
14  20337564373  ACTIVO    HABIDO
Browser closed.
